# Core Silver Transformations

This section transforms the core customer, order, order-item and product entities.

Silver processing includes:

- explicit schema conversion;
- whitespace and blank-value normalization;
- business-key validation;
- business-rule validation;
- duplicate handling;
- quarantine of invalid records;
- source-to-target reconciliation

In [24]:
import uuid
import time
import builtins
from datetime import datetime, timezone
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField ,StringType, LongType, DoubleType, TimestampType


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 26, Finished, Available, Finished, False)

In [25]:
SILVER_RUN_ID = str(uuid.uuid4())
SILVER_STARTED_AT = datetime.now(timezone.utc)
SILVER_START_PERF = time.perf_counter()

LOAD_TYPE = "FULL"

print("Silver run ID: ", SILVER_RUN_ID)
print("Started at: ", SILVER_STARTED_AT)
print("Load type: ", LOAD_TYPE)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 27, Finished, Available, Finished, False)

Silver run ID:  5998e20a-1345-44d5-9602-d00230f7b944
Started at:  2026-07-29 06:40:19.554607+00:00
Load type:  FULL


# Transformation Helpers

Reusable functions standardize text handling, metadata creation and Delta-table writes.

In [26]:
def clean_string(column_name: str):
    """ Trim whitespace and convert blank strings to null. """
    trimmed_value = F.trim(F.col(column_name))
    return F.when(trimmed_value == "",F.lit(None)).otherwise(trimmed_value)

def add_silver_metadata(dataframe: DataFrame, source_table: str) -> DataFrame:
    """ Add technical lineage metadata to a Silver dataframe. """
    return (dataframe
        .withColumn("_silver_run_id", F.lit(SILVER_RUN_ID))
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("_silver_source_table", F.lit(source_table))
        .withColumn("_silver_load_type", F.lit(LOAD_TYPE))
    )

def write_delta_table(dataframe: DataFrame, table_name: str) -> None:
    """ Overwrite a managed Delta table during the static full-load project. """
    dataframe.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(table_name)

def build_quarantine_reason(*conditions):
    """ Combine multiple reason expressions into one semicolon-separated field.
    Each argument must return either a reason string or null."""
    return F.concat_ws("; ", *conditions)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 28, Finished, Available, Finished, False)

In [27]:
silver_audit_records = []

def record_silver_audit (
    *,
    source_table: str,
    target_table: str,
    quarantine_table: str,
    source_row_count: int,
    valid_row_count: int,
    quarantine_row_count: int,
    duplicate_rows_removed: int,
    status: str
) -> None:

    silver_audit_records.append({
        "silver_run_id": SILVER_RUN_ID,
        "source_table": source_table,
        "target_table": target_table,
        "quarantine_table": quarantine_table,
        "load_type": LOAD_TYPE,
        "source_row_count": int(source_row_count),
        "valid_row_count": int(valid_row_count),
        "quarantine_row_count": int(quarantine_row_count),
        "duplicate_rows_removed": int(duplicate_rows_removed),
        "reconciled_row_count": int(valid_row_count + quarantine_row_count + duplicate_rows_removed),
        "row_count_difference": int(source_row_count - valid_row_count - quarantine_row_count - duplicate_rows_removed),
        "load_status": status,
        "processed_at_utc": datetime.now(timezone.utc)
    })


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 29, Finished, Available, Finished, False)

# Customers Transformation

Transform `bronze_customers` into a typed and standardized customer dataset.

In [28]:
bronze_customers = spark.table("bronze_customers")

customers_source_count = bronze_customers.count()

print("Bronze customer rows:",customers_source_count)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 30, Finished, Available, Finished, False)

Bronze customer rows: 99441


In [29]:
customers_prepared = (
    bronze_customers
    .select(
        clean_string("customer_id").alias("customer_id"),
        clean_string("customer_unique_id").alias("customer_unique_id"),
        clean_string("customer_zip_code_prefix").cast("int").alias("customer_zip_prefix"),
        F.upper(clean_string("customer_city")).alias("customer_city"),
        F.upper(clean_string("customer_state")).alias("customer_state"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 31, Finished, Available, Finished, False)

In [30]:
customers_flagged = (
    customers_prepared
    .withColumn("_dq_missing_customer_id", F.col("customer_id").isNull())
    .withColumn("_dq_missing_unique_customer_id", F.col("customer_unique_id").isNull())
    .withColumn("_dq_invalid_zip_prefix", F.col("customer_zip_prefix").isNull())
    .withColumn("_dq_missing_state", F.col("customer_state").isNull())
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 32, Finished, Available, Finished, False)

In [31]:
customers_flagged = (
    customers_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_customer_id"), F.lit("MISSING_CUSTOMER_ID")),
            F.when(F.col("_dq_missing_unique_customer_id"), F.lit("MISSING_CUSTOMER_UNIQUE_ID")),
            F.when(F.col("_dq_invalid_zip_prefix"), F.lit("INVALID_CUSTOMER_ZIP_PREFIX")),
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 33, Finished, Available, Finished, False)

In [32]:
quarantine_customers = (
    customers_flagged
    .filter(
        F.col("_dq_missing_customer_id")
        | F.col("_dq_missing_unique_customer_id")
        | F.col("_dq_invalid_zip_prefix")
    )
)

valid_customers_pre_dedup = (
    customers_flagged
    .filter(
        ~(
            F.col("_dq_missing_customer_id")
            | F.col("_dq_missing_unique_customer_id")
            | F.col("_dq_invalid_zip_prefix")
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 34, Finished, Available, Finished, False)

In [33]:
customer_dedup_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

customers_ranked = (
    valid_customers_pre_dedup
    .withColumn("_duplicate_rank", F.row_number().over(customer_dedup_window))
)

duplicate_customer_rows = (
    customers_ranked
    .filter(F.col("_duplicate_rank") > 1).count()
)

silver_customers = (
    customers_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank", "_quarantine_reason")
)

silver_customers = add_silver_metadata(silver_customers, "bronze_customers")
quarantine_customers = add_silver_metadata(quarantine_customers, "bronze_customers")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 35, Finished, Available, Finished, False)

In [34]:
write_delta_table(silver_customers, "silver_customers")
write_delta_table(quarantine_customers, "quarantine_customers")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 36, Finished, Available, Finished, False)

In [35]:
customers_valid_count = spark.table("silver_customers").count()
customers_quarantine_count = spark.table("quarantine_customers").count()
customers_reconciliation_difference = customers_source_count - customers_valid_count - customers_quarantine_count - duplicate_customer_rows

customers_status = (
    "SUCCESS"
    if customers_reconciliation_difference == 0
    else "FAILED"
)

record_silver_audit(
    source_table = "bronze_customers",
    target_table = "silver_customers",
    quarantine_table = "quarantine_customers",
    source_row_count = customers_source_count,
    valid_row_count = customers_valid_count,
    quarantine_row_count = customers_quarantine_count,
    duplicate_rows_removed = duplicate_customer_rows,
    status = customers_status
)

print(
    "Customers:",
    {
        "source": customers_source_count,
        "valid": customers_valid_count,
        "quarantine": customers_quarantine_count,
        "duplicates_removed": duplicate_customer_rows,
        "status": customers_status
    }
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 37, Finished, Available, Finished, False)

Customers: {'source': 99441, 'valid': 99441, 'quarantine': 0, 'duplicates_removed': 0, 'status': 'SUCCESS'}


# Orders Transformation

Create a typed order table with lifecycle and delivery-quality features.

In [36]:
bronze_orders = spark.table("bronze_orders")

orders_source_count = bronze_orders.count()

orders_prepared = (
    bronze_orders
    .select(
        clean_string("order_id").alias("order_id"),
        clean_string("customer_id").alias("customer_id"),
        F.lower(clean_string("order_status")).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_approved_at").alias("order_approved_ts"),
        F.to_timestamp("order_delivered_carrier_date").alias("carrier_handover_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("delivered_customer_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("estimated_delivery_ts"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 38, Finished, Available, Finished, False)

In [37]:
VALID_ORDER_STATUSES = [
    "approved", "canceled", "created", "delivered", "invoiced", "processing","shipped", "unavailable"
]

orders_flagged = (
    orders_prepared
    .withColumn("_dq_missing_order_id", F.col("order_id").isNull())
    .withColumn("_dq_missing_customer_id", F.col("customer_id").isNull())
    .withColumn("_dq_invalid_order_status", F.col("order_status").isNull() | ~F.col("order_status").isin(VALID_ORDER_STATUSES))
    .withColumn("_dq_missing_purchase_timestamp", F.col("order_purchase_ts").isNull())
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 39, Finished, Available, Finished, False)

In [38]:
orders_flagged = (
    orders_flagged
    .withColumn(
        "_dq_approval_before_purchase",
        F.col("order_approved_ts").isNotNull() & (F.col("order_approved_ts")<F.col("order_purchase_ts"))
    )
    .withColumn(
        "_dq_carrier_before_approval",
        (
            F.col("carrier_handover_ts").isNotNull()
            & F.col("order_approved_ts").isNotNull()
            & (
                F.col("carrier_handover_ts")
                < F.col("order_approved_ts")
            )
        )
    )
    .withColumn(
        "_dq_delivery_before_purchase",
        (
            F.col("delivered_customer_ts").isNotNull()
            & (
                F.col("delivered_customer_ts")
                < F.col("order_purchase_ts")
            )
        )
    )
    .withColumn(
        "_dq_delivered_status_missing_delivery_ts",
        (
            F.col("order_status") == "delivered"
        )
        & F.col(
            "delivered_customer_ts"
        ).isNull()
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 40, Finished, Available, Finished, False)

In [39]:
orders_flagged = (
    orders_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_order_id"), F.lit("MISSING_ORDER_ID")),
            F.when(F.col("_dq_missing_customer_id"), F.lit("MISSING_CUSTOMER_ID")),
            F.when(F.col("_dq_invalid_order_status"), F.lit("INVALID_ORDER_STATUS")),
            F.when(F.col("_dq_missing_purchase_timestamp"),F.lit("MISSING_OR_INVALID_PURCHASE_TIMESTAMP"))
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 41, Finished, Available, Finished, False)

In [40]:
orders_critical_failure = (
    F.col("_dq_missing_order_id")
    | F.col("_dq_missing_customer_id")
    | F.col("_dq_invalid_order_status")
    | F.col("_dq_missing_purchase_timestamp")
)

quarantine_orders = orders_flagged.filter(orders_critical_failure)
valid_orders_pre_dedup = orders_flagged.filter(~orders_critical_failure)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 42, Finished, Available, Finished, False)

In [41]:
order_dedup_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc())
)

orders_ranked = (
    valid_orders_pre_dedup
    .withColumn("_duplicate_rank",F.row_number().over(order_dedup_window))
)

duplicate_order_rows = orders_ranked.filter(F.col("_duplicate_rank") > 1).count()

silver_orders = orders_ranked.filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank","_duplicate_reason")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 43, Finished, Available, Finished, False)

In [42]:
silver_orders = (
    silver_orders
    .withColumn("order_purchase_date",F.to_date("order_purchase_ts"))
    .withColumn("purchase_year",F.year("order_purchase_ts"))
    .withColumn("purchase_month",F.month("order_purchase_ts"))
    .withColumn("purchase_hour",F.hour("order_purchase_ts"))
    .withColumn("purchase_weekday_number",F.dayofweek("order_purchase_ts"))
    .withColumn("purchase_weekday_name",F.date_format("order_purchase_ts","EEEE"))
    .withColumn("is_weekend_purchase",F.when(F.dayofweek("order_purchase_ts").isin(1, 7),True).otherwise(False))
)

silver_orders = (
    silver_orders
    .withColumn(
        "approval_hours",
        (
            F.unix_timestamp(
                "order_approved_ts"
            )
            - F.unix_timestamp(
                "order_purchase_ts"
            )
        ) / F.lit(3600.0)
    )
    .withColumn(
        "carrier_handover_days",
        F.datediff(
            "carrier_handover_ts",
            "order_approved_ts"
        )
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "delivered_customer_ts",
            "order_purchase_ts"
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            "delivered_customer_ts",
            "estimated_delivery_ts"
        )
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("delivered_customer_ts").isNotNull()
            & F.col("estimated_delivery_ts").isNotNull()
            & (
                F.col("delivered_customer_ts")
                > F.col("estimated_delivery_ts")
            ),
            True
        )
        .when(
            F.col("delivered_customer_ts").isNotNull()
            & F.col("estimated_delivery_ts").isNotNull(),
            False
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 44, Finished, Available, Finished, False)

In [43]:
silver_orders = add_silver_metadata(silver_orders, "bronze_orders")
quarantine_orders = add_silver_metadata(quarantine_orders, "bronze_orders")

write_delta_table(silver_orders, "silver_orders")
write_delta_table(quarantine_orders, "quarantine_orders")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 45, Finished, Available, Finished, False)

In [45]:
orders_valid_count = spark.table("silver_orders").count()
orders_quarantine_count = spark.table("quarantine_orders").count()
orders_difference = orders_source_count - orders_valid_count - orders_quarantine_count - duplicate_order_rows
orders_status = "SUCCESS" if orders_difference == 0 else "FAILED"

record_silver_audit(
    source_table = "bronze_orders",
    target_table = "silver_orders",
    quarantine_table = "quarantine_orders",
    source_row_count = orders_source_count,
    valid_row_count = orders_valid_count,
    quarantine_row_count = orders_quarantine_count,
    duplicate_rows_removed = duplicate_order_rows,
    status = orders_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 47, Finished, Available, Finished, False)

# Order-Item Transformation

Create a typed order-item table at one row per order and item sequence.

In [48]:
bronze_order_items = spark.table("bronze_order_items")

order_items_source_count = bronze_order_items.count()

order_items_prepared = (
    bronze_order_items
    .select(
        clean_string("order_id").alias("order_id"),
        clean_string("order_item_id").cast("int").alias("order_item_id"),
        clean_string("product_id").alias("product_id"),
        clean_string("seller_id").alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_ts"),
        clean_string("price").cast("decimal(18,2)").alias("item_price"),
        clean_string("freight_value").cast("decimal(18,2)").alias("freight_value"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 52, Finished, Available, Finished, False)

In [49]:
order_items_flagged = (
    order_items_prepared
    .withColumn("_dq_missing_order_id",F.col("order_id").isNull())
    .withColumn("_dq_invalid_order_item_id",
        F.col("order_item_id").isNull() | (F.col("order_item_id") <= 0)
    )
    .withColumn("_dq_missing_product_id",F.col("product_id").isNull())
    .withColumn("_dq_missing_seller_id",F.col("seller_id").isNull())
    .withColumn("_dq_invalid_item_price",
        F.col("item_price").isNull() | (F.col("item_price") < 0)
    )
    .withColumn("_dq_invalid_freight_value",
        F.col("freight_value").isNull() | (F.col("freight_value") < 0)
    )
)

order_items_critical_failure = (
    F.col("_dq_missing_order_id")
    | F.col("_dq_invalid_order_item_id")
    | F.col("_dq_missing_product_id")
    | F.col("_dq_missing_seller_id")
    | F.col("_dq_invalid_item_price")
    | F.col("_dq_invalid_freight_value")
)

order_items_flagged = (
    order_items_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_order_id"),F.lit("MISSING_ORDER_ID")),
            F.when(F.col("_dq_invalid_order_item_id"),F.lit("INVALID_ORDER_ITEM_ID")),
            F.when(F.col("_dq_missing_product_id"),F.lit("MISSING_PRODUCT_ID")),
            F.when(F.col("_dq_missing_seller_id"),F.lit("MISSING_SELLER_ID")),
            F.when(F.col("_dq_invalid_item_price"),F.lit("INVALID_ITEM_PRICE")),
            F.when(F.col("_dq_invalid_freight_value"),F.lit("INVALID_FREIGHT_VALUE"))
        )
    )
)

quarantine_order_items = order_items_flagged.filter(order_items_critical_failure)

valid_order_items_pre_dedup = order_items_flagged.filter(~order_items_critical_failure)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 53, Finished, Available, Finished, False)

In [50]:
order_item_dedup_window = (
    Window
    .partitionBy("order_id","order_item_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

order_items_ranked = (
    valid_order_items_pre_dedup
    .withColumn("_duplicate_rank",F.row_number().over(order_item_dedup_window))
)

duplicate_order_item_rows = order_items_ranked.filter(F.col("_duplicate_rank") > 1).count()

silver_order_items = (
    order_items_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank","_quarantine_reason")
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 54, Finished, Available, Finished, False)

In [51]:
silver_order_items = (
    silver_order_items
    .withColumn("item_total_value",F.col("item_price") + F.col("freight_value"))
    .withColumn("freight_to_price_ratio",
        F.when(
            F.col("item_price") > 0,
            (F.col("freight_value") / F.col("item_price")).cast("decimal(18,6)")
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 55, Finished, Available, Finished, False)

In [52]:
silver_order_items = add_silver_metadata(silver_order_items,"bronze_order_items")
quarantine_order_items = add_silver_metadata(quarantine_order_items,"bronze_order_items")

write_delta_table(silver_order_items,"silver_order_items")
write_delta_table(quarantine_order_items,"quarantine_order_items")

order_items_valid_count = spark.table("silver_order_items").count()
order_items_quarantine_count = spark.table("quarantine_order_items").count()
order_items_difference = order_items_source_count - order_items_valid_count - order_items_quarantine_count - duplicate_order_item_rows
order_items_status = "SUCCESS" if order_items_difference == 0 else "FAILED"

record_silver_audit(
    source_table="bronze_order_items",
    target_table="silver_order_items",
    quarantine_table="quarantine_order_items",
    source_row_count=order_items_source_count,
    valid_row_count=order_items_valid_count,
    quarantine_row_count=order_items_quarantine_count,
    duplicate_rows_removed=duplicate_order_item_rows,
    status=order_items_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 56, Finished, Available, Finished, False)

# Products Transformation

Create a typed product table with standardized physical attributes and quality flags.

In [54]:
bronze_products = spark.table("bronze_products")

products_source_count = bronze_products.count()

products_prepared = (
    bronze_products
    .select(
        clean_string("product_id").alias("product_id"),
        F.lower(clean_string("product_category_name")).alias("product_category_name_pt"),
        clean_string("product_name_lenght").cast("int").alias("product_name_length"),
        clean_string("product_description_lenght").cast("int").alias("product_description_length"),
        clean_string("product_photos_qty").cast("int").alias("product_photo_count"),
        clean_string("product_weight_g").cast("double").alias("product_weight_g"),
        clean_string("product_length_cm").cast("double").alias("product_length_cm"),
        clean_string("product_height_cm").cast("double").alias("product_height_cm"),
        clean_string("product_width_cm").cast("double").alias("product_width_cm"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

products_flagged = (
    products_prepared
    .withColumn(
        "_dq_missing_product_id",
        F.col("product_id").isNull()
    )
    .withColumn(
        "_dq_missing_category",
        F.col(
            "product_category_name_pt"
        ).isNull()
    )
    .withColumn(
        "_dq_invalid_weight",
        (
            F.col("product_weight_g").isNotNull()
            & (
                F.col("product_weight_g") < 0
            )
        )
    )
    .withColumn(
        "_dq_invalid_dimensions",
        (
            (
                F.col("product_length_cm").isNotNull()
                & (
                    F.col("product_length_cm") < 0
                )
            )
            | (
                F.col("product_height_cm").isNotNull()
                & (
                    F.col("product_height_cm") < 0
                )
            )
            | (
                F.col("product_width_cm").isNotNull()
                & (
                    F.col("product_width_cm") < 0
                )
            )
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 58, Finished, Available, Finished, False)

In [55]:
products_critical_failure = (
    F.col("_dq_missing_product_id")
    | F.col("_dq_invalid_weight")
    | F.col("_dq_invalid_dimensions")
)

products_flagged = (
    products_flagged
    .withColumn("_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_product_id"),F.lit("MISSING_PRODUCT_ID")),
            F.when(F.col("_dq_invalid_weight"),F.lit("INVALID_PRODUCT_WEIGHT")),
            F.when(F.col("_dq_invalid_dimensions"),F.lit("INVALID_PRODUCT_DIMENSIONS"))
        )
    )
)

quarantine_products = products_flagged.filter(products_critical_failure)
valid_products_pre_dedup = products_flagged.filter(~products_critical_failure)

product_dedup_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

products_ranked = (
    valid_products_pre_dedup
    .withColumn("_duplicate_rank",F.row_number().over(product_dedup_window))
)

duplicate_product_rows = products_ranked.filter(F.col("_duplicate_rank") > 1).count()

silver_products = (
    products_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank","_quarantine_reason")
)

silver_products = (
    silver_products
    .withColumn(
        "product_volume_cm3",
        F.when(
            F.col("product_length_cm").isNotNull()
            & F.col("product_height_cm").isNotNull()
            & F.col("product_width_cm").isNotNull(),
            F.col("product_length_cm")
            * F.col("product_height_cm")
            * F.col("product_width_cm")
        )
    )
    .withColumn(
        "product_density_g_per_cm3",
        F.when(
            F.col("product_volume_cm3") > 0,
            F.col("product_weight_g")
            / F.col("product_volume_cm3")
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 59, Finished, Available, Finished, False)

In [56]:
silver_products = add_silver_metadata(silver_products,"bronze_products")
quarantine_products = add_silver_metadata(quarantine_products,"bronze_products")

write_delta_table(silver_products,"silver_products")
write_delta_table(quarantine_products,"quarantine_products")

products_valid_count = spark.table("silver_products").count()
products_quarantine_count = spark.table("quarantine_products").count()
products_difference = products_source_count - products_valid_count - products_quarantine_count - duplicate_product_rows
products_status = "SUCCESS" if products_difference == 0 else "FAILED"

record_silver_audit(
    source_table="bronze_products",
    target_table="silver_products",
    quarantine_table="quarantine_products",
    source_row_count=products_source_count,
    valid_row_count=products_valid_count,
    quarantine_row_count=products_quarantine_count,
    duplicate_rows_removed=duplicate_product_rows,
    status=products_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 60, Finished, Available, Finished, False)

In [59]:
""" Saving Audit Table """

silver_audit_schema = StructType([
    StructField("silver_run_id", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("quarantine_table", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("valid_row_count", LongType(), False),
    StructField("quarantine_row_count", LongType(), False),
    StructField("duplicate_rows_removed", LongType(), False),
    StructField("reconciled_row_count", LongType(), False),
    StructField("row_count_difference", LongType(), False),
    StructField("load_status", StringType(), False),
    StructField("processed_at_utc", TimestampType(), False)
])

silver_audit_df = spark.createDataFrame(silver_audit_records,schema=silver_audit_schema)

silver_audit_df.write.mode("append").format("delta").saveAsTable("audit_silver_load")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 63, Finished, Available, Finished, False)

# Sellers Transformation

Standardize seller identifiers and geographical attributes at one row per seller.

In [68]:
bronze_sellers = spark.table("bronze_sellers")

sellers_source_count = bronze_sellers.count()

sellers_prepared = (
    bronze_sellers
    .select(
        clean_string( "seller_id").alias("seller_id"),
        clean_string("seller_zip_code_prefix").cast("int").alias("seller_zip_prefix"),
        F.upper(clean_string("seller_city")).alias("seller_city"),
        F.upper(clean_string("seller_state")).alias("seller_state"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

sellers_flagged = (
    sellers_prepared
    .withColumn("_dq_missing_seller_id",F.col("seller_id").isNull())
    .withColumn("_dq_invalid_seller_zip",F.col("seller_zip_prefix").isNull())
    .withColumn("_dq_missing_seller_city",F.col("seller_city").isNull())
    .withColumn("_dq_missing_seller_state",F.col("seller_state").isNull())
)

sellers_flagged = (
    sellers_flagged
    .withColumn("_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_seller_id"),F.lit("MISSING_SELLER_ID")),
            F.when(F.col("_dq_invalid_seller_zip"),F.lit("INVALID_SELLER_ZIP_PREFIX"))
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 72, Finished, Available, Finished, False)

In [69]:
sellers_critical_failure = (F.col("_dq_missing_seller_id") | F.col("_dq_invalid_seller_zip"))

quarantine_sellers = sellers_flagged.filter(sellers_critical_failure)
valid_sellers_pre_dedup =  sellers_flagged.filter(~sellers_critical_failure)

seller_dedup_window = (
    Window
    .partitionBy("seller_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

sellers_ranked = (
    valid_sellers_pre_dedup
    .withColumn("_duplicate_rank", F.row_number().over(seller_dedup_window))
)

duplicate_seller_rows = sellers_ranked.filter(F.col("_duplicate_rank") > 1).count()
silver_sellers = sellers_ranked.filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank", "_quarantine_reason")

silver_sellers = add_silver_metadata(silver_sellers, "bronze_sellers")

quarantine_sellers = add_silver_metadata(quarantine_sellers, "bronze_sellers")

write_delta_table(silver_sellers, "silver_sellers")
write_delta_table(quarantine_sellers, "quarantine_sellers")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 73, Finished, Available, Finished, False)

In [70]:
sellers_valid_count = spark.table("silver_sellers").count()
sellers_quarantine_count = spark.table("quarantine_sellers").count()

sellers_difference = (sellers_source_count - sellers_valid_count - sellers_quarantine_count - duplicate_seller_rows)
sellers_status = "SUCCESS" if sellers_difference == 0 else "FAILED"

record_silver_audit(
    source_table="bronze_sellers",
    target_table="silver_sellers",
    quarantine_table="quarantine_sellers",
    source_row_count=sellers_source_count,
    valid_row_count=sellers_valid_count,
    quarantine_row_count=sellers_quarantine_count,
    duplicate_rows_removed=duplicate_seller_rows,
    status=sellers_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 74, Finished, Available, Finished, False)

# Payment Transformation

Create a typed payment table at one row per order and payment sequence.

In [71]:
bronze_payments = spark.table("bronze_order_payments")

payments_source_count = bronze_payments.count()

payments_prepared = (
    bronze_payments
    .select(
        clean_string("order_id").alias("order_id"),
        clean_string("payment_sequential").cast("int").alias("payment_sequence"),
        F.lower(clean_string("payment_type")).alias("payment_type"),
        clean_string("payment_installments").cast("int").alias("payment_installments"),
        clean_string("payment_value").cast("decimal(18,2)").alias("payment_value"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 75, Finished, Available, Finished, False)

In [74]:
VALID_PAYMENT_TYPES = [
    "boleto",
    "credit_card",
    "debit_card",
    "not_defined",
    "voucher"
]

payments_flagged = (
    payments_prepared
    .withColumn(
        "_dq_missing_order_id",
        F.col("order_id").isNull()
    )
    .withColumn(
        "_dq_invalid_payment_sequence",
        (
            F.col("payment_sequence").isNull()
            | (F.col("payment_sequence") <= 0)
        )
    )
    .withColumn(
        "_dq_invalid_payment_type",
        (
            F.col("payment_type").isNull()
            | ~F.col("payment_type").isin(
                VALID_PAYMENT_TYPES
            )
        )
    )
    .withColumn(
        "_dq_invalid_installments",
        (
            F.col("payment_installments").isNull()
            | (
                F.col("payment_installments") < 0
            )
        )
    )
    .withColumn(
        "_dq_invalid_payment_value",
        (
            F.col("payment_value").isNull()
            | (F.col("payment_value") < 0)
        )
    )
)

payments_flagged = (
    payments_flagged
    .withColumn("_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_order_id"), F.lit("MISSING_ORDER_ID")),
            F.when(F.col("_dq_invalid_payment_sequence"), F.lit("INVALID_PAYMENT_SEQUENCE")),
            F.when(F.col("_dq_invalid_payment_type"), F.lit("INVALID_PAYMENT_TYPE")),
            F.when(F.col("_dq_invalid_installments"), F.lit("INVALID_INSTALLMENTS")),
            F.when(F.col("_dq_invalid_payment_value"), F.lit("INVALID_PAYMENT_VALUE"))
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 78, Finished, Available, Finished, False)

In [75]:
payments_critical_failure = (
    F.col("_dq_missing_order_id")
    | F.col("_dq_invalid_payment_sequence")
    | F.col("_dq_invalid_payment_type")
    | F.col("_dq_invalid_installments")
    | F.col("_dq_invalid_payment_value")
)

quarantine_order_payments = payments_flagged.filter(payments_critical_failure)
valid_payments_pre_dedup = payments_flagged.filter(~payments_critical_failure)

payment_dedup_window = (
    Window
    .partitionBy("order_id", "payment_sequence")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

payments_ranked = (
    valid_payments_pre_dedup
    .withColumn("_duplicate_rank", F.row_number().over(payment_dedup_window))
)

duplicate_payment_rows = (
    payments_ranked
    .filter(F.col("_duplicate_rank") > 1)
    .count()
)

silver_order_payments = (
    payments_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank", "_quarantine_reason")
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 79, Finished, Available, Finished, False)

In [76]:
silver_order_payments = (
    silver_order_payments
    .withColumn("uses_installments",
        F.when(F.col("payment_installments") > 1, True).otherwise(False)
    )
    .withColumn("payment_installment_bucket",
        F.when( F.col("payment_installments") <= 1, F.lit("1"))
        .when(F.col("payment_installments") <= 3, F.lit("2-3"))
        .when(F.col("payment_installments") <= 6, F.lit("4-6"))
        .when(F.col("payment_installments") <= 12, F.lit("7-12"))
        .otherwise(F.lit("13+"))
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 80, Finished, Available, Finished, False)

In [77]:
silver_order_payments = add_silver_metadata(silver_order_payments, "bronze_order_payments")
quarantine_order_payments = add_silver_metadata(quarantine_order_payments, "bronze_order_payments")

write_delta_table(silver_order_payments, "silver_order_payments")
write_delta_table(quarantine_order_payments, "quarantine_order_payments")

payments_valid_count = spark.table("silver_order_payments").count()
payments_quarantine_count = spark.table("quarantine_order_payments").count()
payments_difference = payments_source_count - payments_valid_count - payments_quarantine_count - duplicate_payment_rows
payments_status = "SUCCESS" if payments_difference == 0 else "FAILED"

record_silver_audit(
    source_table = "bronze_order_payments",
    target_table = "silver_order_payments",
    quarantine_table = "quarantine_order_payments",
    source_row_count = payments_source_count,
    valid_row_count = payments_valid_count,
    quarantine_row_count = payments_quarantine_count,
    duplicate_rows_removed = duplicate_payment_rows,
    status = payments_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 81, Finished, Available, Finished, False)

# Review Transformation

Create a cleaned review table at one row per review identifier while retaining review text and timestamps.

In [80]:
bronze_reviews = spark.table("bronze_order_reviews")

reviews_source_count = bronze_reviews.count()

reviews_prepared = (
    bronze_reviews
    .select(
        clean_string("review_id").alias("review_id"),
        clean_string("order_id").alias("order_id"),
        clean_string("review_score").cast("int").alias("review_score"),
        clean_string("review_comment_title").alias("review_title"),
        clean_string("review_comment_message").alias("review_comment"),
        F.to_timestamp("review_creation_date").alias("review_created_ts"),
        F.to_timestamp("review_answer_timestamp").alias("review_answered_ts"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

reviews_flagged = (
    reviews_prepared
    .withColumn(
        "_dq_missing_review_id",
        F.col("review_id").isNull()
    )
    .withColumn(
        "_dq_missing_order_id",
        F.col("order_id").isNull()
    )
    .withColumn(
        "_dq_invalid_review_score",
        (
            F.col("review_score").isNull()
            | ~F.col(
                "review_score"
            ).between(1, 5)
        )
    )
    .withColumn(
        "_dq_missing_review_created_ts",
        F.col("review_created_ts").isNull()
    )
    .withColumn(
        "_dq_answer_before_creation",
        (
            F.col("review_answered_ts").isNotNull()
            & F.col("review_created_ts").isNotNull()
            & (
                F.col("review_answered_ts")
                < F.col("review_created_ts")
            )
        )
    )
)

reviews_flagged = (
    reviews_flagged
    .withColumn("_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_review_id"),F.lit("MISSING_REVIEW_ID")),
            F.when(F.col("_dq_missing_order_id"),F.lit("MISSING_ORDER_ID")),
            F.when(F.col("_dq_invalid_review_score"),F.lit("INVALID_REVIEW_SCORE")),
            F.when(F.col("_dq_missing_review_created_ts"),F.lit("MISSING_REVIEW_CREATION_TIMESTAMP"))
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 84, Finished, Available, Finished, False)

In [81]:
reviews_critical_failure = (
    F.col("_dq_missing_review_id")
    | F.col("_dq_missing_order_id")
    | F.col("_dq_invalid_review_score")
    | F.col("_dq_missing_review_created_ts")
)

quarantine_order_reviews = reviews_flagged.filter(reviews_critical_failure)
valid_reviews_pre_dedup = reviews_flagged.filter(~reviews_critical_failure)

review_dedup_window = (
    Window
    .partitionBy("review_id")
    .orderBy(
        F.col("review_answered_ts").desc_nulls_last(),
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

reviews_ranked = (
    valid_reviews_pre_dedup
    .withColumn("_duplicate_rank", F.row_number().over(review_dedup_window))
)

duplicate_review_rows = (
    reviews_ranked
    .filter( F.col("_duplicate_rank") > 1)
    .count()
)

silver_order_reviews = (
    reviews_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank","_quarantine_reason")
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 85, Finished, Available, Finished, False)

In [82]:
silver_order_reviews = (
    silver_order_reviews
    .withColumn("has_review_title", F.col("review_title").isNotNull())
    .withColumn("has_review_comment", F.col("review_comment").isNotNull())
    .withColumn("review_comment_length", F.length("review_comment"))
    .withColumn("review_response_hours",
        (
            F.unix_timestamp("review_answered_ts") - F.unix_timestamp("review_created_ts")
        ) / F.lit(3600.0)
    )
    .withColumn("review_score_group",
        F.when(F.col("review_score") <= 2,F.lit("Low"))
        .when(F.col("review_score") == 3, F.lit("Neutral"))
        .otherwise(F.lit("High"))
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 86, Finished, Available, Finished, False)

In [83]:
silver_order_reviews = add_silver_metadata(silver_order_reviews, "bronze_order_reviews")
quarantine_order_reviews = add_silver_metadata(quarantine_order_reviews, "bronze_order_reviews")

write_delta_table(silver_order_reviews,"silver_order_reviews")
write_delta_table(quarantine_order_reviews,"quarantine_order_reviews")

reviews_valid_count = spark.table("silver_order_reviews").count()
reviews_quarantine_count = spark.table("quarantine_order_reviews").count()

reviews_difference = reviews_source_count - reviews_valid_count - reviews_quarantine_count - duplicate_review_rows
reviews_status = "SUCCESS" if reviews_difference == 0 else "FAILED"

record_silver_audit(
    source_table = "bronze_order_reviews",
    target_table = "silver_order_reviews",
    quarantine_table = "quarantine_order_reviews",
    source_row_count = reviews_source_count,
    valid_row_count = reviews_valid_count,
    quarantine_row_count = reviews_quarantine_count,
    duplicate_rows_removed = duplicate_review_rows,
    status = reviews_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 87, Finished, Available, Finished, False)

# Category Translation Transformation

Create a unique Portuguese-to-English product-category reference table.

In [84]:
bronze_translation = spark.table("bronze_category_translation")

translation_source_count = bronze_translation.count()

translation_prepared = (
    bronze_translation
    .select(
        F.lower(clean_string("product_category_name")).alias("product_category_name_pt"),
        F.lower(clean_string("product_category_name_english")).alias("product_category_name_en"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 88, Finished, Available, Finished, False)

In [85]:
translation_flagged = (
    translation_prepared
    .withColumn("_dq_missing_portuguese_category",
        F.col("product_category_name_pt").isNull()
    )
    .withColumn("_dq_missing_english_category",
        F.col("product_category_name_en").isNull()
    )
)

translation_flagged = (
    translation_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(
                F.col("_dq_missing_portuguese_category"),
                F.lit("MISSING_PORTUGUESE_CATEGORY")
            ),
            F.when(
                F.col("_dq_missing_english_category"),
                F.lit("MISSING_ENGLISH_CATEGORY")
            )
        )
    )
)

translation_critical_failure = (
    F.col("_dq_missing_portuguese_category")
    | F.col("_dq_missing_english_category")
)

quarantine_category_translation = translation_flagged.filter(translation_critical_failure)
valid_translation_pre_dedup = translation_flagged.filter(~translation_critical_failure)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 89, Finished, Available, Finished, False)

In [86]:
translation_dedup_window = (
    Window
    .partitionBy(
        "product_category_name_pt"
    )
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("product_category_name_en").asc()
    )
)

translation_ranked = (
    valid_translation_pre_dedup
    .withColumn("_duplicate_rank", F.row_number().over(translation_dedup_window))
)

duplicate_translation_rows = (
    translation_ranked
    .filter(F.col("_duplicate_rank") > 1)
    .count()
)

silver_category_translation = (
    translation_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank", "_quarantine_reason")
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 90, Finished, Available, Finished, False)

In [87]:
silver_category_translation = add_silver_metadata(silver_category_translation,"bronze_category_translation")
quarantine_category_translation = add_silver_metadata(quarantine_category_translation,"bronze_category_translation")

write_delta_table(silver_category_translation,"silver_category_translation")
write_delta_table(quarantine_category_translation,"quarantine_category_translation")

translation_valid_count = spark.table("silver_category_translation").count()
translation_quarantine_count = spark.table("quarantine_category_translation").count()
translation_difference = translation_source_count - translation_valid_count - translation_quarantine_count - duplicate_translation_rows
translation_status = "SUCCESS" if translation_difference == 0 else "FAILED"

record_silver_audit(
    source_table = "bronze_category_translation",
    target_table = "silver_category_translation",
    quarantine_table = "quarantine_category_translation",
    source_row_count = translation_source_count,
    valid_row_count = translation_valid_count,
    quarantine_row_count = translation_quarantine_count,
    duplicate_rows_removed = duplicate_translation_rows,
    status = translation_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 91, Finished, Available, Finished, False)

# Product Category Enrichment

Enrich Silver products with cleaned English category names while preserving the original Portuguese category.

In [89]:
silver_products_base = spark.table("silver_products")

translation_lookup = spark.table("silver_category_translation").select("product_category_name_pt","product_category_name_en")

silver_products_enriched = (
    silver_products_base.alias("p")
    .join(
        translation_lookup.alias("t"),
        on="product_category_name_pt",
        how="left"
    )
    .withColumn(
        "product_category_name_en",
        F.coalesce(
            F.col(
                "product_category_name_en"
            ),
            F.lit("unknown")
        )
    )
    .withColumn(
        "_dq_missing_category_translation",
        (
            F.col("product_category_name_pt")
            .isNotNull()
            & (
                F.col(
                    "product_category_name_en"
                )
                == "unknown"
            )
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 93, Finished, Available, Finished, False)

In [90]:
write_delta_table(silver_products_enriched, "silver_products")

products_before_enrichment = silver_products_base.count()
products_after_enrichment = spark.table("silver_products").count()

if products_before_enrichment != products_after_enrichment:
    raise RuntimeError(
        "Product category enrichment changed "
        "the product row count."
    )

print("Product category enrichment passed:", products_after_enrichment)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 94, Finished, Available, Finished, False)

Product category enrichment passed: 32951


# Geolocation Transformation

Create a cleaned geolocation table and an aggregated ZIP-prefix lookup for customer and seller enrichment.

In [91]:
bronze_geolocation = spark.table("bronze_geolocation")

geolocation_source_count = bronze_geolocation.count()

geolocation_prepared = (
    bronze_geolocation
    .select(
        clean_string("geolocation_zip_code_prefix").cast("int").alias("zip_prefix"),
        clean_string("geolocation_lat").cast("double").alias("latitude"),
        clean_string("geolocation_lng").cast("double").alias("longitude"),
        F.upper(clean_string("geolocation_city")).alias("city"),
        F.upper(clean_string("geolocation_state")).alias("state"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

geolocation_flagged = (
    geolocation_prepared
    .withColumn("_dq_invalid_zip_prefix",F.col("zip_prefix").isNull())
    .withColumn("_dq_invalid_latitude",
        (
            F.col("latitude").isNull()
            | ~F.col("latitude").between(-90.0, 90.0)
        )
    )
    .withColumn("_dq_invalid_longitude",
        (
            F.col("longitude").isNull()
            | ~F.col("longitude").between(-180.0, 180.0)
        )
    )
    .withColumn("_dq_missing_city",F.col("city").isNull())
    .withColumn("_dq_missing_state", F.col("state").isNull())
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 95, Finished, Available, Finished, False)

In [92]:
geolocation_flagged = (
    geolocation_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col( "_dq_invalid_zip_prefix"),F.lit("INVALID_ZIP_PREFIX")),
            F.when(F.col("_dq_invalid_latitude"),F.lit("INVALID_LATITUDE")),
            F.when(F.col("_dq_invalid_longitude"),F.lit("INVALID_LONGITUDE"))
        )
    )
)

geolocation_critical_failure = (
    F.col("_dq_invalid_zip_prefix")
    | F.col("_dq_invalid_latitude")
    | F.col("_dq_invalid_longitude")
)

quarantine_geolocation = geolocation_flagged.filter(geolocation_critical_failure)
valid_geolocation_pre_dedup = geolocation_flagged.filter(~geolocation_critical_failure)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 96, Finished, Available, Finished, False)

In [93]:
geolocation_key_columns = [
    "zip_prefix",
    "latitude",
    "longitude",
    "city",
    "state"
]

geolocation_dedup_window = (
    Window
    .partitionBy(*geolocation_key_columns)
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

geolocation_ranked = (
    valid_geolocation_pre_dedup
    .withColumn("_duplicate_rank",
        F.row_number().over(geolocation_dedup_window)
    )
)

duplicate_geolocation_rows = (
    geolocation_ranked
    .filter(F.col("_duplicate_rank") > 1)
    .count()
)

silver_geolocation = (
    geolocation_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank", "_quarantine_reason")
)

silver_geolocation = add_silver_metadata(silver_geolocation,"bronze_geolocation")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 97, Finished, Available, Finished, False)

In [94]:
zip_location_counts = (
    silver_geolocation
    .groupBy("zip_prefix", "city", "state")
    .agg(F.count("*").alias("location_frequency"))
)

zip_location_window = (
    Window
    .partitionBy("zip_prefix")
    .orderBy(
        F.col("location_frequency").desc(),
        F.col("state").asc_nulls_last(),
        F.col("city").asc_nulls_last()
    )
)

zip_primary_location = (
    zip_location_counts
    .withColumn("_location_rank", F.row_number().over(zip_location_window))
    .filter(F.col("_location_rank") == 1)
    .select("zip_prefix", 
        F.col("city").alias("primary_city"),
        F.col("state").alias("primary_state"),
        "location_frequency"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 98, Finished, Available, Finished, False)

In [95]:
zip_coordinate_summary = (
    silver_geolocation
    .groupBy("zip_prefix")
    .agg(
        F.avg("latitude").alias("average_latitude"),
        F.avg("longitude").alias("average_longitude"),
        F.count("*").alias("coordinate_record_count"),
        F.countDistinct(F.struct("latitude","longitude")).alias("distinct_coordinate_count"),
        F.countDistinct("city").alias("distinct_city_count"),
        F.countDistinct("state").alias("distinct_state_count")
    )
)

silver_geolocation_zip = (
    zip_coordinate_summary
    .join(zip_primary_location, on="zip_prefix", how="left")
    .withColumn("_silver_run_id", F.lit(SILVER_RUN_ID))
    .withColumn("_silver_processed_at", F.current_timestamp())
    .withColumn("_silver_source_table", F.lit("silver_geolocation"))
    .withColumn("_silver_load_type", F.lit(LOAD_TYPE))
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 99, Finished, Available, Finished, False)

In [96]:
quarantine_geolocation = add_silver_metadata(quarantine_geolocation,"bronze_geolocation")

write_delta_table(silver_geolocation, "silver_geolocation")
write_delta_table(silver_geolocation_zip, "silver_geolocation_zip")
write_delta_table(quarantine_geolocation, "quarantine_geolocation")

geolocation_valid_count = spark.table("silver_geolocation").count()
geolocation_quarantine_count = spark.table("quarantine_geolocation").count()
geolocation_difference = geolocation_source_count - geolocation_valid_count - geolocation_quarantine_count - duplicate_geolocation_rows
geolocation_status = "SUCCESS" if geolocation_difference == 0 else "FAILED"

record_silver_audit(
    source_table = "bronze_geolocation",
    target_table = "silver_geolocation",
    quarantine_table = "quarantine_geolocation",
    source_row_count = geolocation_source_count,
    valid_row_count = geolocation_valid_count,
    quarantine_row_count = geolocation_quarantine_count,
    duplicate_rows_removed = duplicate_geolocation_rows,
    status = geolocation_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 100, Finished, Available, Finished, False)

# Silver Referential-Integrity Validation

Validate the cleaned parent-child relationships after type conversion, filtering and deduplication.

In [97]:
SILVER_RELATIONSHIPS = [
    {
        "relationship_id": "SREL001",
        "relationship_name": "Silver Orders to Customers",
        "child_table": "silver_orders",
        "child_key": "customer_id",
        "parent_table": "silver_customers",
        "parent_key": "customer_id",
        "severity": "ERROR"
    },
    {
        "relationship_id": "SREL002",
        "relationship_name": "Silver Items to Orders",
        "child_table": "silver_order_items",
        "child_key": "order_id",
        "parent_table": "silver_orders",
        "parent_key": "order_id",
        "severity": "ERROR"
    },
    {
        "relationship_id": "SREL003",
        "relationship_name": "Silver Items to Products",
        "child_table": "silver_order_items",
        "child_key": "product_id",
        "parent_table": "silver_products",
        "parent_key": "product_id",
        "severity": "ERROR"
    },
    {
        "relationship_id": "SREL004",
        "relationship_name": "Silver Items to Sellers",
        "child_table": "silver_order_items",
        "child_key": "seller_id",
        "parent_table": "silver_sellers",
        "parent_key": "seller_id",
        "severity": "ERROR"
    },
    {
        "relationship_id": "SREL005",
        "relationship_name": "Silver Payments to Orders",
        "child_table": "silver_order_payments",
        "child_key": "order_id",
        "parent_table": "silver_orders",
        "parent_key": "order_id",
        "severity": "ERROR"
    },
    {
        "relationship_id": "SREL006",
        "relationship_name": "Silver Reviews to Orders",
        "child_table": "silver_order_reviews",
        "child_key": "order_id",
        "parent_table": "silver_orders",
        "parent_key": "order_id",
        "severity": "WARNING"
    }
]

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 101, Finished, Available, Finished, False)

In [98]:
silver_relationship_records = []

for relationship in SILVER_RELATIONSHIPS:

    child_df = (
        spark.table(relationship["child_table"])
        .select(F.col(relationship["child_key"]).alias("relationship_key"))
        .filter(F.col("relationship_key").isNotNull())
    )

    parent_df = (
        spark.table(relationship["parent_table"])
        .select(F.col(relationship["parent_key"]).alias("relationship_key"))
        .filter(F.col("relationship_key").isNotNull())
        .dropDuplicates()
    )

    child_row_count = child_df.count()

    orphan_df = child_df.join(parent_df, on="relationship_key", how="left_anti")
    orphan_row_count = orphan_df.count()

    silver_relationship_records.append({
        "silver_run_id": SILVER_RUN_ID,
        "relationship_id": (relationship["relationship_id"]),
        "relationship_name": (relationship["relationship_name"]),
        "child_table": (relationship["child_table"]),
        "child_key": (relationship["child_key"]),
        "parent_table": (relationship["parent_table"]),
        "parent_key": (relationship["parent_key"]),
        "severity": (relationship["severity"]),
        "evaluated_child_row_count": int(child_row_count),
        "orphan_row_count": int(orphan_row_count),
        "orphan_percentage": (
            builtins.round(orphan_row_count / child_row_count * 100, 4)
            if child_row_count > 0 else 0.0
        ),
        "status": "PASS" if orphan_row_count == 0 else "FAIL",
        "tested_at_utc": datetime.now(timezone.utc)
    })

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 102, Finished, Available, Finished, False)

In [99]:
silver_relationship_df = spark.createDataFrame(silver_relationship_records)

display(silver_relationship_df.orderBy("relationship_id"))

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 103, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e595826f-2e4a-45f9-9e54-e5e9c0d39a32)

In [100]:
silver_relationship_df.write.mode("append").format("delta").saveAsTable("audit_silver_relationship")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 104, Finished, Available, Finished, False)

In [101]:
critical_relationship_failures = (
    silver_relationship_df
    .filter(
        (F.col("severity") == "ERROR")
        & (F.col("status") == "FAIL")
    )
    .count()
)

if critical_relationship_failures > 0:
    raise RuntimeError(
        "Critical Silver referential-integrity "
        "checks failed."
    )

print("Silver relationship validation passed.")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 105, Finished, Available, Finished, False)

Silver relationship validation passed.


In [102]:
EXPECTED_SILVER_TARGETS = {
    "silver_customers",
    "silver_orders",
    "silver_order_items",
    "silver_products",
    "silver_sellers",
    "silver_order_payments",
    "silver_order_reviews",
    "silver_category_translation",
    "silver_geolocation"
}

actual_silver_targets = {record["target_table"] for record in silver_audit_records}
missing_silver_audits = EXPECTED_SILVER_TARGETS - actual_silver_targets

if missing_silver_audits:
    raise RuntimeError(
        "Missing Silver audits for: "
        f"{missing_silver_audits}"
    )

print(
    "All expected Silver entities "
    "are represented in the audit."
)

silver_audit_df = spark.createDataFrame(silver_audit_records,schema=silver_audit_schema)

current_run_existing_count = (
    spark.table("audit_silver_load")
    .filter(F.col("silver_run_id") == SILVER_RUN_ID)
    .count()
    if spark.catalog.tableExists("audit_silver_load") else 0
)

if current_run_existing_count == 0:
    silver_audit_df.write.mode("append").format("delta").saveAsTable("audit_silver_load")
else:
    print(
        "Audit rows for this Silver run "
        "already exist; append skipped."
    )

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 106, Finished, Available, Finished, False)

All expected Silver entities are represented in the audit.
Audit rows for this Silver run already exist; append skipped.


In [103]:
SILVER_COMPLETED_AT = datetime.now(timezone.utc)
SILVER_DURATION_SECONDS = builtins.round(time.perf_counter() - SILVER_START_PERF, 2)

failed_silver_tables = [
    record for record in silver_audit_records if record["load_status"] != "SUCCESS"
]

silver_run_history_record = [{
    "silver_run_id": SILVER_RUN_ID,
    "started_at_utc": SILVER_STARTED_AT,
    "completed_at_utc": SILVER_COMPLETED_AT,
    "duration_seconds": SILVER_DURATION_SECONDS,
    "load_type": LOAD_TYPE,
    "source_table_count": len(silver_audit_records),
    "successful_table_count": sum(record["load_status"] == "SUCCESS" for record in silver_audit_records),
    "failed_table_count": len(failed_silver_tables),
    "critical_relationship_failure_count": int(critical_relationship_failures),
    "status": "SUCCESS" if not failed_silver_tables and critical_relationship_failures == 0 else "FAILED"
}]

silver_run_history_df = spark.createDataFrame(silver_run_history_record)
display(silver_run_history_df)
silver_run_history_df.write.mode("append").format("delta").saveAsTable("audit_silver_run_history")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 107, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56cf7624-a695-4719-8c72-af7497048c1f)

In [104]:
display(
    spark.table("audit_silver_load")
    .filter(
        F.col("silver_run_id")
        == SILVER_RUN_ID
    )
    .orderBy("target_table")
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 108, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0f3ef090-e11e-45d8-a766-ecf7e2ceab44)

In [105]:
display(
    spark.table("silver_products")
    .select(
        "product_id",
        "product_category_name_pt",
        "product_category_name_en",
        "_dq_missing_category_translation"
    )
    .limit(20)
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 109, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 001774e4-e16d-4a8b-836d-1493fb889017)

In [106]:
display(
    spark.table(
        "silver_geolocation_zip"
    )
    .orderBy(
        F.col(
            "coordinate_record_count"
        ).desc()
    )
    .limit(20)
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 110, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12e60b1d-f181-4497-9c7a-4bd06d9d2df0)

In [107]:
display(
    spark.table(
        "silver_order_payments"
    )
    .select(
        "order_id",
        "payment_sequence",
        "payment_type",
        "payment_installments",
        "payment_value",
        "uses_installments",
        "payment_installment_bucket"
    )
    .limit(20)
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 111, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d07a4efe-f74a-4c62-bed0-f96a67a9166e)

In [108]:
display(
    spark.table(
        "silver_order_reviews"
    )
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_score_group",
        "has_review_comment",
        "review_comment_length"
    )
    .limit(20)
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 112, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe4077e5-0584-4405-bcb9-2d2c856e2957)